# Dealing with EPIC Out-of-Time Events -- Part 1: Images
<hr style="border: 2px solid #fadbac" />

- **Description:** This thread will allow the user to create an image cleaned from out-of-time events.
- **Level:** Intermediate
- **Data:** XMM observation of the Circinus Galaxy (obsid=0111240101)
- **Requirements:** Must be run using pySAS version 2.2.8 or higher.
- **Credit:** Ryan Tanner (December 2025), based on an <a href="https://www.cosmos.esa.int/web/xmm-newton/sas-thread-epic-oot">ESA SOC SAS Tutorial</a>
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 5 December 2025, for SAS v22.1 and pySAS v2.0

<hr style="border: 2px solid #fadbac" />

## 1. Introduction

In EPIC imaging modes, photons are not only registered during the actual integration interval, but also during the readout of a CCD. These so called Out-of-Time (OoT) events are assigned incorrect RAWY values, leading to a wrong energy correction. OoT events broaden the spectral features, and create a strip of events with wrongly reconstructed position. The fraction of OoT events scales with the (mode-dependent) ratio of integration and readout time, and is highest for pn Full Frame (6.3%) and Extended Full Frame (2.3%) modes (the user is referred to the [XMM-Newton Users Handbook](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/uhb/epicoot.html) for more details). Examples of the effects of OoT events in pn images and spectra are shown in the [User Guide to the XMM-Newton Science Analysis System](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/epicOoT.html). It is important to stress that for most targets a correction of OoT events in the spectrum is not necessary. In any case, a correction is **only necessary** if OoT events overlap the source being investigated.

The general process for dealing with out-of-time contamination is:

1. Calibrate and filter a normal event list.
2. Process an out-of-time event list using the same method.
3. Scale the out-of-time event list based on the out-of-time fraction.
4. Subtract the scaled out-of-time events from the normal event list.

### Chains vs. Procs (`epchain` vs. `epproc`)

In other tutorials we use the "procs" (`epproc` and `emproc`) to generate calibrated event lists. In this tutorial we use the "chains" (`epchain` and `emchain`). The chains do all the same things as the procs, plus more. The filenames for event lists generated by the chains is different than the filenames generated by the procs. pySAS can automatically detect either type of event list.

Event lists generated by the procs have the following general filename structure:

`YYYY_XXXObsIDXX_INST_SSSS_ImagingEvts.ds`

where

- YYYY: The revolution number
- XXXObsIDXX: The observation identifier
- INST: The instrument name, EPN or EMOS1 or EMOS2
- SSSS: The exposure identifier. For instance: S001

Event lists generated by the chains have the following general filename structure:

`PXXXObsIDXXZZSSSSZZEVLI0000.FIT`

where

- XXXObsIDXX: The observation identifier
- ZZ: The instrument name, PN or M1 or M2
- SSSS: The exposure identifier. For instance: S001
- ZZ: An identifier depending on the EPIC mode, and/or whether `withoutoftime` and `withctisrcpos` are set

When either the procs or the chains are run they will silently overwrite any previously generated event lists.

#### SAS Tasks to be Used

- `epchain`[(Documentation for epchain)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/epchain/index.html "epchain Documentation")

#### Useful Links

- [`pysas` Documentation](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/pysas/index.html "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads/ "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html "Helpdesk") - Link to form to contact the GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

## 2. Setup

In [ ]:
# pySAS imports
import pysas
from pysas import MyTask

# Useful imports
import re

# HEASoftpy import
import heasoftpy as hsp

# Astropy imports
from astropy.io import fits

# To handle certain warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
obsid  = '0111240101'
my_obs = pysas.ObsID(obsid)

***
The cell below contains a number of functions that will be used throughout this notebook.

In [ ]:
def filter_event_list(in_event_list,
                      out_event_list = 'filtered_event_list.fits',
                      pi_min  = 200,
                      pi_max  = 13000,
                      pattern = None):

    with fits.open(in_event_list) as hdu:
        instrument = hdu[0].header['INSTRUME']

    if instrument == 'EPN':
        filter = 'XMMEA_EP'
        if pattern is None: pattern = 4
    elif 'EMOS' in instrument:
        filter = 'XMMEA_EM'
        if pattern is None: pattern = 12

    # Filter expression
    expression = '(PATTERN in [0:{pattern}])&&(PI in [{pi_min}:{pi_max}])&&(FLAG == 0)&&#{filter}'.format(filter=filter,pattern=pattern,pi_min=pi_min,pi_max=pi_max)

    inargs = {'table'           : in_event_list, 
              'withfilteredset' : 'yes', 
              "expression"      : expression, 
              'filteredset'     : out_event_list, 
              'filtertype'      : 'expression', 
              'keepfilteroutput': 'yes', 
              'updateexposure'  : 'yes', 
              'filterexposure'  : 'yes'}
    
    MyTask('evselect', inargs).run()

## 3. Run 'epchain'

We will run `epchain` twice. Once to generate a standard event list, and then again to generate the out of time event list.

<div class="alert alert-block alert-info">
    <b>Note:</b> You can use either <tt>basic_setup</tt> or <tt>MyTask</tt> to run <tt>epchain</tt>. Internally <tt>basic_setup</tt> uses <tt>MyTask</tt> to run <tt>epchain</tt>, but it includes some checks to see if it has already been run and will not run it again unless the option '<tt>rerun=True</tt>' is passed into <tt>basic_setup</tt>. Using <tt>MyTask</tt> will silently overwrite any previously generated event lists.
</div>

In [ ]:
inargs = {'runatthkgen'   : False, 
          'runepframes'   : False, 
          'runbadpixfind' : False,
          'runbadpix'     : False}

my_obs.basic_setup(overwrite    = False,
                   rerun        = True,
                   run_epchain  = True,
                   epchain_args = inargs,
                   run_emproc   = False,
                   run_rgsproc  = False)

This will run `epchain` with `withoutoftime=True`. This will **not** overwrite the previously generated event lists because the event lists made with `withoutoftime` have a slightly different file name (see below).

In [ ]:
inargs = {'runbackground'    : False, 
          'keepintermediate' : 'raw', 
          'withoutoftime'    : True}

MyTask('epchain', inargs).run()

my_obs.find_event_list_files(print_output=False)

Now let's look at the pn event lists we have.

In [ ]:
for file in my_obs.files['PNevt_list']: print(file)

The filenames of the event lists generated by the first run of `epchain` will look like this:

P0111240101PNS003**PI**EVLI0000.FIT

The filenames of the event lists generated by the second run of `epchain`, with `withoutoftime=True`, will look like this:

P0111240101PNS003**OO**EVLI0000.FIT

They are the exact same except the first has "**PI**" in the name, and the second has "**OO**" in the name. The next cell will store the filenames in the variables `event_list_file` and `outoftime_file`. It will also set up filenames that will be used throughout this notebook.

In [ ]:
for filename in my_obs.files['PNevt_list']:
    if re.search('.*PN.*PIEVLI.*FIT$',filename):
        event_list_file = filename
    if re.search('.*PN.*OOEVLI.*FIT$',filename):
        outoftime_file = filename

filtered_evtli_file = 'PN_clean_event_list_file.fits'
filtered_oot_file   = 'PN_clean_outoftime_file.fits'

pn_obs_image    = 'PN_observation_image.fits'
pn_oot_image    = 'PN_OoT_image.fits'
pn_oot_rescaled = 'PN_OoT_image_rescaled.fits'
pn_clean_image  = 'PN_observation_clean_image.fits'

Now we do some basic filtering to make clean event lists.

In [ ]:
filter_event_list(event_list_file, out_event_list=filtered_evtli_file)

In [ ]:
filter_event_list(outoftime_file, out_event_list=filtered_oot_file)

## 4. Generate Cleaned Images

Let's take a look at the images of the two event lists.

In [ ]:
# "Normal" event list
my_obs.quick_eplot(filtered_evtli_file, image_file=pn_obs_image, vmin=1.0, vmax=1000.0)

In [ ]:
# Out of time event list
my_obs.quick_eplot(filtered_oot_file, image_file=pn_oot_image, vmin=1.0, vmax=1000.0)

Well, that's interesting. Let's see if we can find the difference between them. For this we will use [`farith`](https://heasarc.gsfc.nasa.gov/docs/software/lheasoft/help/farith.html) from HEASoft to first scale the out-of-time image, and then subtract it from the normal event list image.

The out-of-time image is scaled by a factor of 0.063 (6.3%) because this is the expected fraction of out-of-time events for the pn in Full Frame Mode. The fraction of out-of-time events depends on the camera mode, and is also different for the MOS cameras (the user is referred to the [XMM-Newton Users Handbook](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/uhb/epicoot.html) for more details).

In [ ]:
hsp.farith(infil1   = pn_oot_image,
           infil2   = 0.063,
           outfil   = pn_oot_rescaled,
           ops      = 'MUL',
           noprompt = True,
           clobber  = True)

An alternative to using `farith` would be to use `ftimgcalc`.

```python
hsp.ftimgcalc(outfile=pn_oot_rescaled, expr='A*0.063', a=pn_oot_image)
```

In [ ]:
my_obs.quick_implot(pn_oot_rescaled)

Now we can subtract the rescaled out-of-time image from the original image to produce a clean image. We will compare the two images below.

In [ ]:
hsp.farith(infil1   = pn_obs_image,
           infil2   = pn_oot_rescaled,
           outfil   = pn_clean_image,
           ops      = 'SUB',
           noprompt = True,
           clobber  = True)

In [ ]:
my_obs.quick_implot(pn_obs_image, vmin=1.0, vmax=1000.0)
my_obs.quick_implot(pn_clean_image, vmin=1.0, vmax=1000.0)

We see that in the cleaned image the stripe caused by the bright source in the center has been removed. We now have a clean image with the out-of-time events removed.